# CineEmbed — 08 Round 2: z-sweep on the AE family

Round 2 of the locked two-round modeling strategy, retargeted (2026-05-17 amendment) from DEC family to AE family. The demo backbone is `ae_z64`; this notebook trains `ae_z32` and `ae_z128` to test z-dim sensitivity.

Self-contained. Designed to run on a fresh Colab account with no prior cineembed install. Aggressively clears caches, re-clones the repo, force-reinstalls the package, and verifies imports before any training begins. Skip-if-done aware (won't re-train if artifacts already exist on Drive).

Recipe — identical to MVP `ae_z64`. Cold-start multi-modal AE with W2 inverse-variance weights, 100 epochs max, early-stop patience 10, lr=1e-3, batch_size=512, seed=42, gradient_clip_norm=1.0, `hidden_dim=128` held constant across z (only `latent_dim` varies). Note: at z=128 the FC backbone becomes 154 → 128 → 128 (square last layer); honest comparison knob, documented.

**Configs (~30 min total on T4):**

| run_name | latent_dim | hidden_dim |
|---|---:|---:|
| `ae_z32` | 32 | 128 |
| `ae_z64` (MVP carry-over) | 64 | 128 |
| `ae_z128` | 128 | 128 |

**Prerequisites:**
- Google Drive folder `MyDrive/CineEmbed/artifacts/` populated with at least:
  - `feature_matrix.npz` (production feature matrix, ~384 MB)
  - `movies_eda_final.csv` (~241 MB)
  - `models/ae_z64.pt` (MVP AE checkpoint — needed for the summary's z=64 row; if missing the summary cell prints a warning and skips that row)

If `ae_z64.pt` is present but `inference/ae_z64/manifest.json` is missing, the notebook builds the missing manifest inline before training begins (so the final summary table can compare all three z values consistently).

All outputs land under `artifacts/models/<run>/` (training) and `artifacts/inference/<run>/` (inference index) on Drive.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Aggressive setup — works on a fresh Colab runtime with NO prior state.
# Clears stale module imports, removes any cached repo, re-clones, force-
# reinstalls, and verifies that the imports actually have the latest API
# before letting training proceed.
# ════════════════════════════════════════════════════════════════════════════
import os, sys, json, time, hashlib, shutil, importlib, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# --- Step 1: Drive mount (Colab only) ----------------------------------------
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = Path('/content/cineembed-repo')
    ARTIFACTS = Path('/content/drive/MyDrive/CineEmbed/artifacts')
else:
    REPO_ROOT = Path('..').resolve()
    ARTIFACTS = REPO_ROOT / 'artifacts'

# --- Step 2: defensively purge stale Python state ----------------------------
for m in list(sys.modules):
    if m.startswith('cineembed'):
        del sys.modules[m]
        print(f'  [purge] sys.modules[{m}]')

if IN_COLAB:
    # Remove cached pip-installed cineembed (any version)
    subprocess.run(['pip', 'uninstall', 'cineembed', '-y', '-q'],
                    stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

    # Drop any cached __pycache__ inside an existing repo dir
    if REPO_ROOT.exists():
        for pyc in REPO_ROOT.rglob('__pycache__'):
            shutil.rmtree(pyc, ignore_errors=True)
        for pyc in REPO_ROOT.rglob('*.pyc'):
            try:
                pyc.unlink()
            except FileNotFoundError:
                pass

    # --- Step 3: ensure fresh clone of the feature branch --------------------
    # Prefer GITHUB_TOKEN if user added it to Colab Secrets (private-repo support).
    try:
        from google.colab import userdata
        _token = userdata.get('GITHUB_TOKEN')
        REPO_URL = f'https://{_token}@github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git'
    except Exception:
        REPO_URL = 'https://github.com/bkaankaya/CineEmbed-A-Multi-Modal-Unsupervised-Film-Recommendation-System.git'

    if not REPO_ROOT.exists():
        get_ipython().system(f'git clone {REPO_URL} {REPO_ROOT}')
    else:
        get_ipython().system(
            f'cd {REPO_ROOT} && git fetch -q --all && '
            f'git checkout main -q && '
            f'git reset --hard origin/main -q && '
            f'git pull -q'
        )

    # --- Step 4: force editable install ---------------------------------------
    get_ipython().system(
        f'pip install -e {REPO_ROOT} --force-reinstall --no-deps -q'
    )
    # Project deps for retrieval-eval parquet output + wandb
    get_ipython().system('pip install wandb pyarrow -q')

# --- Step 5: put repo src/ first on sys.path ---------------------------------
src_dir = str(REPO_ROOT / 'src')
if src_dir in sys.path:
    sys.path.remove(src_dir)
sys.path.insert(0, src_dir)

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nRepo:      {REPO_ROOT}')
print(f'Artifacts: {ARTIFACTS}')
print(f'Device:    {DEVICE}')

if IN_COLAB:
    get_ipython().system(f'cd {REPO_ROOT} && git log --oneline -1')

## Verify imports — bail loud if anything is missing

This cell confirms that the freshly-cloned `cineembed` package contains the latest API surface (in particular `train.train_model` must accept `wandb_run`, and `cineembed.wandb_integration` must exist). If either check fails, **do not** run any subsequent cell — go back to the setup cell, restart the runtime, and re-run from the top.

In [ ]:
import importlib
import inspect

# Re-import after purge
if 'cineembed' in sys.modules:
    importlib.reload(sys.modules['cineembed'])

from cineembed import data, backbone, heads, losses, train, eval as cev
from cineembed.wandb_integration import wandb_run as _wandb_run
from cineembed.wandb_integration import log_epoch as _log_epoch
from cineembed.wandb_integration import log_eval as _log_eval
from cineembed.wandb_integration import log_artifact as _log_artifact

# Probe the train.train_model signature
tm_params = list(inspect.signature(train.train_model).parameters)
print(f'train.train_model params: {tm_params}')
assert 'wandb_run' in tm_params, (
    'STALE INSTALL: train.train_model is missing the wandb_run param. '
    'Re-run the setup cell, then Runtime → Restart runtime, then re-run from top.'
)

print(f'\ntrain.train_model:           OK  (wandb_run accepted)')
print(f'cineembed.wandb_integration: OK')
print(f'cineembed module path:       {data.__file__}')

# Final guardrails — fail loud if the production data is missing
for p in [ARTIFACTS / 'feature_matrix.npz', ARTIFACTS / 'movies_eda_final.csv']:
    assert p.exists(), (
        f'MISSING REQUIRED ARTIFACT: {p}\n'
        f'Populate Drive with the EDA outputs before running Round 2.'
    )
print('feature_matrix.npz + movies_eda_final.csv: OK')

# ae_z64 carry-over is optional but flagged
ae_z64_ckpt = ARTIFACTS / 'models' / 'ae_z64.pt'
ae_z64_manifest = ARTIFACTS / 'inference' / 'ae_z64' / 'manifest.json'
if not ae_z64_ckpt.exists():
    print(f'WARN: {ae_z64_ckpt} missing — z=64 row will be omitted from summary.')
elif not ae_z64_manifest.exists():
    print(f'WARN: {ae_z64_manifest} missing — rebuilding inline below.')
else:
    print('ae_z64 carry-over: OK')

## W&B configuration (auto-detected — no interactive prompt)

Detection ladder: Colab Secret `WANDB_API_KEY` → process env `WANDB_API_KEY` → `~/.netrc`. Falls back to `WANDB_MODE=offline` if all three miss; never prompts stdin.

In [ ]:
WANDB_PROJECT = 'cineembed'
WANDB_ENTITY  = None
WANDB_GROUP   = 'round-2'

WANDB_MODE = None
WANDB_KEY_PRESENT = False

if WANDB_PROJECT and IN_COLAB:
    try:
        from google.colab import userdata
        _key = userdata.get('WANDB_API_KEY')
        if _key:
            os.environ['WANDB_API_KEY'] = _key
            WANDB_KEY_PRESENT = True
    except Exception:
        pass

if WANDB_PROJECT and not WANDB_KEY_PRESENT and os.environ.get('WANDB_API_KEY'):
    WANDB_KEY_PRESENT = True

if WANDB_PROJECT and not WANDB_KEY_PRESENT and not IN_COLAB:
    if Path.home().joinpath('.netrc').exists():
        WANDB_KEY_PRESENT = True

if WANDB_PROJECT is None:
    WANDB_MODE = 'disabled'
    print('W&B disabled.')
elif WANDB_KEY_PRESENT:
    WANDB_MODE = 'online'
    os.environ['WANDB_MODE'] = 'online'
    print(f"W&B ONLINE  project='{WANDB_PROJECT}'  group='{WANDB_GROUP}'")
else:
    WANDB_MODE = 'offline'
    os.environ['WANDB_MODE'] = 'offline'
    print('No WANDB_API_KEY found → OFFLINE mode.')
    print(f'Offline runs land under {REPO_ROOT}/wandb/; sync later with `wandb sync`.')

print(f'\nFinal: WANDB_MODE={WANDB_MODE}')

## Data + globals
Single load, reused across all sweep variants. Same shape as MVP / Phase 1 / Round 1.

In [ ]:
import numpy as np
import pandas as pd

X, feature_names = data.load_feature_matrix(ARTIFACTS / 'feature_matrix.npz')
labels = data.get_labels(ARTIFACTS / 'movies_eda_final.csv')
block_slices = data.get_block_indices(feature_names)
has_bio = X[:, block_slices['director'].start + 64].clone()

BLOCK_DIMS = {b: (slc.stop - slc.start) for b, slc in block_slices.items()}
PROJ_DIMS = backbone.DEFAULT_PROJ_DIMS
w_blocks = losses.compute_block_weights(X.numpy(), block_slices, w_min=0.1, w_max=10.0)

train_idx, val_idx = data.train_val_split(X.shape[0], val_frac=0.1, seed=42)
train_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=True,
                                     indices=train_idx, block_slices=block_slices, seed=42)
val_loader = data.make_dataloader(X, has_bio, batch_size=512, shuffle=False,
                                    indices=val_idx, block_slices=block_slices, seed=42)

print(f'X: {X.shape}  has_bio_sum: {int(has_bio.sum())}')
print(f'BLOCK_DIMS: {BLOCK_DIMS}')

## Helpers — training, encoding, evaluation, manifest writing

All mirror existing patterns: `02_train_ae.ipynb` for AE training, `scripts/train_contrastive.py` for clustering eval shape, `scripts/build_index.py` for retrieval eval shape.

In [ ]:
from contextlib import nullcontext


def _build_backbone(latent_dim, hidden_dim=128, dropout=0.2):
    return backbone.MultiModalBackbone(
        BLOCK_DIMS,
        proj_dims=PROJ_DIMS,
        hidden_dim=hidden_dim,
        latent_dim=latent_dim,
        dropout=dropout,
    )


def train_ae_zN(run_name, latent_dim, *, n_epochs=100, wandb_run=None):
    torch.manual_seed(42)
    bb = _build_backbone(latent_dim=latent_dim)
    ae_head = heads.AEHead(bb, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)

    def loss_fn(model, batch):
        decoded = model(batch['blocks'])
        return losses.weighted_recon_loss(decoded, batch['blocks'],
                                          batch['has_bio'], w_blocks)

    ckpt = ARTIFACTS / 'models' / run_name / 'ae.pt'
    ckpt.parent.mkdir(parents=True, exist_ok=True)

    history = train.train_model(
        model=ae_head, loss_fn=loss_fn,
        train_loader=train_loader, val_loader=val_loader,
        n_epochs=n_epochs, lr=1e-3, weight_decay=1e-5,
        early_stop_patience=10, early_stop_min_delta=1e-4,
        gradient_clip_norm=1.0,
        checkpoint_path=ckpt, device=DEVICE, seed=42,
        wandb_run=wandb_run,
    )
    train.load_checkpoint(ae_head, ckpt, device=DEVICE)
    return ae_head, history


@torch.no_grad()
def encode_all_raw(ae_head, device):
    """Forward all 329k rows through the backbone (raw, unnormalised). For KMeans/GMM eval."""
    ae_head.eval()
    full_loader = data.make_dataloader(X, has_bio, batch_size=2048, shuffle=False,
                                         block_slices=block_slices)
    chunks = []
    for batch in full_loader:
        blk = {k_: v.to(device) for k_, v in batch['blocks'].items()}
        z = ae_head.encode(blk)
        chunks.append(z.cpu().numpy())
    return np.concatenate(chunks, axis=0).astype(np.float32)


def l2_normalize_rows(z):
    norms = np.linalg.norm(z, axis=1, keepdims=True)
    norms = np.where(norms < 1e-8, 1.0, norms)
    return (z / norms).astype(np.float32)


def do_clustering_eval(z, *, seed=42):
    """KMeans + GMM + per-axis-k + multilabel macro-NMI."""
    kmeans_assign = cev.cluster_assignments_kmeans(z, k=21, seed=seed)
    try:
        gmm_assign = cev.cluster_assignments_gmm(z, k=21, seed=seed)
        gmm_eval = cev.evaluate_run(gmm_assign, labels)
    except Exception as e:
        print(f'  [warn] GMM failed (likely angular collapse): {e}')
        gmm_eval = {'_error': str(e)}

    out = {
        'kmeans_k21':        cev.evaluate_run(kmeans_assign, labels),
        'gmm_k21':           gmm_eval,
        'per_axis_k_kmeans': cev.evaluate_run_per_axis_k(z, labels, seed=seed),
    }
    g = block_slices['genre']
    genre_onehot = X[:, g.start:g.start + 21].numpy().astype(np.float32)
    out['multilabel_genre_macro_nmi_kmeans'] = cev.multilabel_macro_nmi(
        kmeans_assign, genre_onehot, metric='nmi')
    return out


def _parse_release_year(s):
    if not isinstance(s, str):
        return None
    try:
        return int(s.split('/')[-1])
    except (ValueError, IndexError):
        return None


def build_films_table_inline(artifacts_dir, expected_rows):
    csv_path = artifacts_dir / 'movies_eda_final.csv'
    df = pd.read_csv(csv_path, low_memory=False)
    assert len(df) == expected_rows, (
        f'Row mismatch: csv has {len(df)} rows but encoder produced {expected_rows}.'
    )
    keep = ['id', 'imdb_id', 'title', 'original_title', 'release_date',
            'director_name', 'genres', 'overview', 'popularity',
            'vote_average', 'vote_count', 'runtime', 'original_language']
    keep = [c for c in keep if c in df.columns]
    out = df.loc[:, keep].copy()
    if 'release_date' in out.columns:
        out['year'] = pd.Series(out['release_date']).apply(_parse_release_year)
    if 'genres' in out.columns:
        out['genres'] = pd.Series(out['genres']).fillna('').apply(
            lambda s: [g for g in s.split('|') if g] if isinstance(s, str) else [])
    return out


EYEBALL_QUERIES = [
    'Inception', 'The Godfather', 'Toy Story', 'The Shawshank Redemption',
    'Pulp Fiction', 'The Matrix', 'Interstellar', 'Forrest Gump',
    'The Dark Knight', 'Spirited Away',
]


def do_retrieval_eval(z_norm, films_df, *, k=5, n_queries=500, seed=42):
    rng = np.random.default_rng(seed)
    n = z_norm.shape[0]
    query_idx = rng.choice(n, size=min(n_queries, n), replace=False)
    primary_genre = np.asarray(
        films_df['genres'].apply(lambda gs: gs[0] if gs else '').values
    )

    genre_at_k_scores = []
    for qi in query_idx:
        if not primary_genre[qi]:
            continue
        sims = z_norm @ z_norm[qi]
        top = np.argpartition(-sims, k + 1)[: k + 1]
        top = top[np.argsort(-sims[top])]
        neighbors = top[top != qi][:k]
        same = int((primary_genre[neighbors] == primary_genre[qi]).sum())
        genre_at_k_scores.append(same / k)

    pair_a = rng.choice(n, size=5000, replace=False)
    pair_b = rng.choice(n, size=5000, replace=False)
    pair_a = pair_a[pair_a != pair_b]
    pair_b = pair_b[: len(pair_a)]
    pair_cos = (z_norm[pair_a] * z_norm[pair_b]).sum(axis=1)

    return {
        'k': k,
        'n_queries': len(genre_at_k_scores),
        'genre_at_k_mean':   float(np.mean(genre_at_k_scores)) if genre_at_k_scores else None,
        'genre_at_k_median': float(np.median(genre_at_k_scores)) if genre_at_k_scores else None,
        'genre_at_k_std':    float(np.std(genre_at_k_scores)) if genre_at_k_scores else None,
        'angular': {
            'random_pair_cos_mean': float(pair_cos.mean()),
            'random_pair_cos_std':  float(pair_cos.std()),
            'random_pair_cos_min':  float(pair_cos.min()),
            'random_pair_cos_max':  float(pair_cos.max()),
        },
    }


def do_eyeball(z_norm, films_df, *, k=5):
    print()
    print(f'Eyeball top-{k} for well-known queries:')
    print('=' * 72)
    rows = []
    titles_lower = films_df['title'].astype(str).str.lower()
    for q in EYEBALL_QUERIES:
        ql = q.lower()
        exact = films_df.index[titles_lower == ql].tolist()
        if exact:
            qi = max(exact, key=lambda i: films_df.iloc[i].get('popularity', 0) or 0)
        else:
            contains = films_df.index[titles_lower.str.contains(ql, regex=False, na=False)].tolist()
            if not contains:
                print(f'  ?? "{q}"  — no match')
                continue
            qi = max(contains, key=lambda i: films_df.iloc[i].get('popularity', 0) or 0)
        sims = z_norm @ z_norm[qi]
        top = np.argpartition(-sims, k + 1)[: k + 1]
        top = top[np.argsort(-sims[top])]
        neighbors = [int(i) for i in top if int(i) != int(qi)][:k]
        q_row = films_df.iloc[qi]
        q_label = f"{q_row['title']} ({q_row.get('year') or '?'})"
        print(f'\n  Q: {q_label}')
        for j, ni in enumerate(neighbors, 1):
            n_row = films_df.iloc[ni]
            g_str = '|'.join(n_row['genres'][:3]) if n_row['genres'] else ''
            print(f'     {j}. {n_row["title"]} ({n_row.get("year") or "?"})  '
                  f'[{g_str}]  cos={sims[ni]:.3f}')
        rows.append({
            'query': q,
            'matched_title': q_row['title'],
            'matched_year': int(q_row['year']) if pd.notna(q_row.get('year')) else None,
            'neighbors': [
                {
                    'title': films_df.iloc[int(ni)]['title'],
                    'year': int(films_df.iloc[int(ni)].get('year'))
                            if pd.notna(films_df.iloc[int(ni)].get('year')) else None,
                    'cosine': float(sims[int(ni)]),
                }
                for ni in neighbors
            ],
        })
    print('=' * 72)
    return rows


def write_inference_index(z_norm, films_df, eval_retrieval, eyeball_rows, *,
                          run_name, latent_dim, hidden_dim, infer_dir, ckpt_path,
                          model_type='ae', extra_meta=None):
    np.save(infer_dir / 'embeddings.npy', z_norm)
    films_df.to_parquet(infer_dir / 'films.parquet', index=False)
    sha = hashlib.sha256(Path(ckpt_path).read_bytes()).hexdigest()[:32]
    manifest = {
        'schema_version':       1,
        'created_unix_seconds': int(time.time()),
        'checkpoint':           str(ckpt_path),
        'checkpoint_sha256_32': sha,
        'model_type':           model_type,
        'latent_dim':           latent_dim,
        'hidden_dim':           hidden_dim,
        'n_clusters':           None,
        'n_films':              int(z_norm.shape[0]),
        'embedding_dim':        int(z_norm.shape[1]),
        'normalization':        'L2',
        'distance_metric':      'cosine (dot product after L2 normalization)',
        'retrieval':            eval_retrieval,
        'eyeball':              eyeball_rows,
    }
    if extra_meta:
        manifest.update(extra_meta)
    (infer_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2))


def push_round2_to_wandb(run, eval_clustering, eval_retrieval, run_dir, run_name):
    if run is None:
        return
    _log_eval(run, eval_clustering['kmeans_k21'], prefix='km_')
    gm = eval_clustering.get('gmm_k21', {})
    if isinstance(gm, dict) and 'genre_nmi' in gm:
        _log_eval(run, gm, prefix='gmm_', add_geo_nmi=False)
    pa = eval_clustering.get('per_axis_k_kmeans', {})
    if pa:
        _log_eval(run, pa, prefix='axis_', add_geo_nmi=False)
    ml = eval_clustering.get('multilabel_genre_macro_nmi_kmeans', {})
    if isinstance(ml, dict) and 'macro_nmi' in ml:
        run.log({'km_multilabel_macro_nmi': float(ml['macro_nmi'])})

    if eval_retrieval:
        run.log({
            'retrieval_genre_at_5_mean':   eval_retrieval.get('genre_at_k_mean'),
            'retrieval_genre_at_5_median': eval_retrieval.get('genre_at_k_median'),
            'retrieval_genre_at_5_std':    eval_retrieval.get('genre_at_k_std'),
            'retrieval_pair_cos_mean':     eval_retrieval['angular']['random_pair_cos_mean'],
            'retrieval_pair_cos_std':      eval_retrieval['angular']['random_pair_cos_std'],
        })
        run.summary['headline_genre_at_5_mean'] = eval_retrieval.get('genre_at_k_mean')

    km = eval_clustering['kmeans_k21']
    if all(kk in km for kk in ('genre_nmi','decade_nmi','lang_nmi')):
        prod = max(km['genre_nmi']*km['decade_nmi']*km['lang_nmi'], 0.0)
        run.summary['headline_km_geo_nmi'] = prod**(1/3) if prod > 0 else 0.0
        run.summary['headline_km_genre_nmi'] = float(km['genre_nmi'])

    for fn in ('ae.pt', 'backbone.pt'):
        p = run_dir / fn
        if p.exists():
            _log_artifact(run, p, name=f'{fn[:-3]}__{run_name}', type='model')


print('Helpers defined: train_ae_zN, encode_all_raw, l2_normalize_rows, '
      'do_clustering_eval, do_retrieval_eval, do_eyeball, '
      'build_films_table_inline, write_inference_index, push_round2_to_wandb')

## Auto-build the `ae_z64` inference index if it's missing on this Drive

If the new Google account's Drive doesn't have `artifacts/inference/ae_z64/manifest.json` (because that index was generated on the previous account or locally on a Mac), build it inline now so the final summary table can put all three z rows side-by-side. No-op if the manifest is already present.

In [ ]:
ae_z64_ckpt = ARTIFACTS / 'models' / 'ae_z64.pt'
ae_z64_infer_dir = ARTIFACTS / 'inference' / 'ae_z64'
ae_z64_manifest = ae_z64_infer_dir / 'manifest.json'

if not ae_z64_ckpt.exists():
    print(f'Skip — {ae_z64_ckpt} not present on this Drive. '
          f'Summary will omit the z=64 row.')
elif ae_z64_manifest.exists():
    print(f'Skip — {ae_z64_manifest} already present.')
else:
    print(f'Building ae_z64 inference index → {ae_z64_infer_dir}/ …')
    ae_z64_infer_dir.mkdir(parents=True, exist_ok=True)

    # Reload the MVP AE
    _bb64 = _build_backbone(latent_dim=64)
    _ae64 = heads.AEHead(_bb64, BLOCK_DIMS, PROJ_DIMS, hidden_dim=128)
    train.load_checkpoint(_ae64, ae_z64_ckpt, device=DEVICE)
    _ae64 = _ae64.to(DEVICE).eval()

    _z64_raw = encode_all_raw(_ae64, DEVICE)
    _z64_norm = l2_normalize_rows(_z64_raw)

    _films64 = build_films_table_inline(ARTIFACTS, expected_rows=_z64_norm.shape[0])
    _ret64 = do_retrieval_eval(_z64_norm, _films64)
    _eye64 = do_eyeball(_z64_norm, _films64)

    write_inference_index(
        _z64_norm, _films64, _ret64, _eye64,
        run_name='ae_z64', latent_dim=64, hidden_dim=128,
        infer_dir=ae_z64_infer_dir, ckpt_path=ae_z64_ckpt,
        model_type='ae', extra_meta={'phase': 'mvp-carryover', 'family': 'ae'},
    )
    print(f"  ae_z64 manifest written. genre@5 mean = {_ret64['genre_at_k_mean']:.3f}")

    # Free memory before sweep
    del _bb64, _ae64, _z64_raw, _z64_norm, _films64
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Sweep — `ae_z32` + `ae_z128`
Skip-if-done aware (won't retrain if all output artifacts exist). Set `FORCE=True` to retrain. Expected ~12 min per config on T4.

In [ ]:
ROUND2_CONFIGS = [
    {'run_name': 'ae_z32',  'latent_dim':  32, 'hidden_dim': 128},
    {'run_name': 'ae_z128', 'latent_dim': 128, 'hidden_dim': 128},
]

FORCE = False

def _all_artifacts_exist(run_dir, infer_dir):
    return (
        (run_dir / 'ae.pt').exists() and
        (run_dir / 'backbone.pt').exists() and
        (run_dir / 'eval.json').exists() and
        (infer_dir / 'manifest.json').exists() and
        (infer_dir / 'embeddings.npy').exists()
    )


for cfg in ROUND2_CONFIGS:
    run_name   = cfg['run_name']
    latent_dim = cfg['latent_dim']
    run_dir    = ARTIFACTS / 'models' / run_name
    infer_dir  = ARTIFACTS / 'inference' / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    infer_dir.mkdir(parents=True, exist_ok=True)

    print('\n' + '#' * 78)
    print(f"# {run_name}  (latent_dim={latent_dim}, mode={WANDB_MODE})")
    print('#' * 78)

    if (not FORCE) and _all_artifacts_exist(run_dir, infer_dir):
        print('SKIP — all artifacts present. Set FORCE=True to overwrite.')
        continue

    _t_run = time.time()

    wandb_cfg = {
        'phase':       'round-2',
        'family':      'ae',
        'latent_dim':  latent_dim,
        'hidden_dim':  cfg['hidden_dim'],
        'batch_size':  512,
        'lr':          1e-3,
        'weight_decay': 1e-5,
    }
    if WANDB_PROJECT:
        ctx = _wandb_run(
            config=wandb_cfg, run_name=run_name,
            project=WANDB_PROJECT, entity=WANDB_ENTITY, group=WANDB_GROUP,
            tags=['round-2', 'ae', f'z{latent_dim}'], mode=WANDB_MODE,
            notes=f'Round 2 z-sweep (AE family). z={latent_dim}.',
        )
    else:
        ctx = nullcontext(None)

    with ctx as run:
        # ── train ──
        ae_head, history = train_ae_zN(run_name, latent_dim=latent_dim, wandb_run=run)
        torch.save(ae_head.backbone.state_dict(), run_dir / 'backbone.pt')
        with open(run_dir / 'history.json', 'w') as f:
            json.dump(history, f, indent=2)

        # ── encode ──
        print(f'\n[encode] forwarding {X.shape[0]} films through backbone…')
        z_raw = encode_all_raw(ae_head, DEVICE)
        z_norm = l2_normalize_rows(z_raw)
        print(f'[encode] z_raw: {z_raw.shape}  '
              f'L2_norm_mean={np.linalg.norm(z_norm, axis=1).mean():.4f}')

        # ── clustering eval ──
        print('[eval] clustering metrics …')
        eval_clustering = do_clustering_eval(z_raw)
        with open(run_dir / 'eval.json', 'w') as f:
            json.dump(eval_clustering, f, indent=2)
        km = eval_clustering['kmeans_k21']
        print(f"[eval] KMeans k=21    g={km['genre_nmi']:.3f}  d={km['decade_nmi']:.3f}  l={km['lang_nmi']:.3f}")

        # ── retrieval eval + inference index ──
        print('[retrieval] building films table + retrieval eval …')
        films_df = build_films_table_inline(ARTIFACTS, expected_rows=z_norm.shape[0])
        eval_retrieval = do_retrieval_eval(z_norm, films_df)
        eyeball_rows = do_eyeball(z_norm, films_df)

        write_inference_index(
            z_norm, films_df, eval_retrieval, eyeball_rows,
            run_name=run_name, latent_dim=latent_dim, hidden_dim=cfg['hidden_dim'],
            infer_dir=infer_dir, ckpt_path=run_dir / 'ae.pt',
            model_type='ae', extra_meta={'phase': 'round-2', 'family': 'ae'},
        )

        print(f"\n[summary] {run_name}: "
              f"genre@5={eval_retrieval['genre_at_k_mean']:.3f}  "
              f"km_gNMI={km['genre_nmi']:.3f}")

        push_round2_to_wandb(run, eval_clustering, eval_retrieval, run_dir, run_name)
        if run is not None:
            print(f'[wandb] run URL: {run.url}')

    # Free up GPU memory between sweep variants
    del ae_head, z_raw, z_norm, films_df
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    elapsed = time.time() - _t_run
    print(f'[{run_name}] total wall-clock: {elapsed/60:.1f} min')

## Round 2 summary — z-sweep table

Three rows: `ae_z32`, `ae_z64` (MVP carry-over), `ae_z128`. Sorted by `genre@5 mean` (demo task), with `geo_NMI` as a clustering-side reference.

If `genre@5` and `geo_NMI` disagree on the winner, **trust `genre@5`** — the deployed task is cosine retrieval, not clustering.

In [ ]:
def _geo_nmi(d):
    if not isinstance(d, dict):
        return None
    g = d.get('genre_nmi'); dd = d.get('decade_nmi'); l = d.get('lang_nmi')
    if None in (g, dd, l):
        return None
    prod = max(g * dd * l, 0.0)
    return prod ** (1/3) if prod > 0 else 0.0


def _load_clustering_eval_for(run_name):
    per_run = ARTIFACTS / 'models' / run_name / 'eval.json'
    if per_run.exists():
        e = json.loads(per_run.read_text())
        return e.get('kmeans_k21')
    flat = ARTIFACTS / 'eval' / 'results.json'
    if flat.exists():
        all_results = json.loads(flat.read_text())
        if run_name in all_results:
            r = all_results[run_name]
            return {k: r[k] for k in r if k.endswith(('_nmi', '_ari')) and isinstance(r[k], (int, float))}
    return None


def _load_retrieval_for(run_name):
    p = ARTIFACTS / 'inference' / run_name / 'manifest.json'
    if not p.exists():
        return None
    m = json.loads(p.read_text())
    return m.get('retrieval')


rows = []
for run_name, z_dim in [('ae_z32', 32), ('ae_z64', 64), ('ae_z128', 128)]:
    km = _load_clustering_eval_for(run_name)
    ret = _load_retrieval_for(run_name)
    rows.append({
        'run':            run_name,
        'z':              z_dim,
        'gNMI':           km.get('genre_nmi') if km else None,
        'dNMI':           km.get('decade_nmi') if km else None,
        'lNMI':           km.get('lang_nmi') if km else None,
        'geo_NMI':        _geo_nmi(km),
        'genre@5 mean':   ret.get('genre_at_k_mean') if ret else None,
        'genre@5 median': ret.get('genre_at_k_median') if ret else None,
        'pair_cos_std':   ret['angular'].get('random_pair_cos_std') if (ret and 'angular' in ret) else None,
    })

df = pd.DataFrame(rows)
fmt = {c: (lambda v: f'{v:.3f}' if isinstance(v, (int, float)) else str(v))
       for c in df.columns if c not in ('run', 'z')}
print(df.to_string(index=False, formatters=fmt))

valid = [r for r in rows if r['genre@5 mean'] is not None]
if valid:
    winner = max(valid, key=lambda r: (r['genre@5 mean'] or 0, r['geo_NMI'] or 0))
    print(f"\n=== Round 2 WINNER by genre@5: {winner['run']}  z={winner['z']}  "
          f"genre@5={winner['genre@5 mean']:.3f}  geo_NMI={winner['geo_NMI']:.3f} ===")
    if winner['run'] != 'ae_z64':
        print(f'  → swap demo backbone to {winner["run"]}')
    else:
        print('  → demo backbone stays ae_z64.')

if WANDB_MODE == 'offline':
    print(f'\nSync offline runs later with:  wandb sync {REPO_ROOT}/wandb/offline-run-*')

## Diagnostic — latent geometry per z

Latent dim-wise std (mean / min) and random-pair cosine spread per backbone. Detects dim collapse and angular spread pathologies.

In [ ]:
print(f"{'run':14}  z   dim_std_mean  dim_std_min  pair_cos_mean  pair_cos_std")
print('-' * 78)
for run_name, z_dim in [('ae_z32', 32), ('ae_z64', 64), ('ae_z128', 128)]:
    p = ARTIFACTS / 'inference' / run_name / 'embeddings.npy'
    if not p.exists():
        print(f'{run_name:14}  {z_dim:>3}  (embeddings.npy missing)')
        continue
    z = np.load(p)
    std_dim = z.std(axis=0)
    manifest_p = ARTIFACTS / 'inference' / run_name / 'manifest.json'
    if manifest_p.exists():
        m = json.loads(manifest_p.read_text())
        ang = m.get('retrieval', {}).get('angular', {})
        pair_mean = ang.get('random_pair_cos_mean')
        pair_std  = ang.get('random_pair_cos_std')
    else:
        pair_mean = pair_std = None
    pm = f'{pair_mean:>13.3f}' if pair_mean is not None else f'{"n/a":>13}'
    ps = f'{pair_std:>12.3f}' if pair_std is not None else f'{"n/a":>12}'
    print(f'{run_name:14}  {z_dim:>3}  {std_dim.mean():>12.4f}  {std_dim.min():>11.6f}  {pm}  {ps}')

## Next steps

1. If `genre@5` winner ≠ `ae_z64`, the deployed inference index in `artifacts/inference/<winner>/` is already on Drive (this notebook wrote it). Backend / frontend teams just point at the new directory.
2. Append the three Round 2 rows to `docs/journal/10-results-table.md`. Cross-reference `docs/journal/07-retrieval-vs-nmi-discovery.md` if `genre@5` and `geo_NMI` disagree.
3. Final report — add the three-row z-sweep sub-table after the Round 1 architecture comparison table.
4. Demo deck — one slide on z-sensitivity with the same table.
5. No Round 3 planned. Further sweeping is explicitly future work — see `docs/journal/08-scope-cuts-future-work.md`.